In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding,
    Attention, Concatenate
)
from tensorflow.keras.models import Model

english_sentences = [
    "hello",
    "how are you",
    "what is your name",
    "where are you going",
    "i am going to college",
    "i like tamil",
    "i am learning artificial intelligence",
    "good morning",
    "thank you",
    "see you tomorrow",
    "what are you doing",
    "i am fine",
    "this is my book",
    "i love my family",
    "where do you live"
]

tamil_sentences = [
    "வணக்கம்",
    "நீங்கள் எப்படி இருக்கிறீர்கள்",
    "உங்கள் பெயர் என்ன",
    "நீங்கள் எங்கே செல்கிறீர்கள்",
    "நான் கல்லூரிக்கு செல்கிறேன்",
    "எனக்கு தமிழ் பிடிக்கும்",
    "நான் செயற்கை நுண்ணறிவை கற்றுக்கொண்டிருக்கிறேன்",
    "காலை வணக்கம்",
    "நன்றி",
    "நாளை சந்திப்போம்",
    "நீங்கள் என்ன செய்கிறீர்கள்",
    "நான் நன்றாக இருக்கிறேன்",
    "இது என்னுடைய புத்தகம்",
    "நான் என் குடும்பத்தை நேசிக்கிறேன்",
    "நீங்கள் எங்கே வசிக்கிறீர்கள்"
]

tamil_sentences = [
    "<start> " + sentence + " <end>"
    for sentence in tamil_sentences
]


# English tokenizer
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

# Tamil tokenizer
# filters='' is IMPORTANT
# It keeps <start> and <end> tokens
tam_tokenizer = Tokenizer(filters='')
tam_tokenizer.fit_on_texts(tamil_sentences)

# Convert sentences to numbers
eng_sequences = eng_tokenizer.texts_to_sequences(
    english_sentences
)

tam_sequences = tam_tokenizer.texts_to_sequences(
    tamil_sentences
)

max_eng_len = max(
    len(sequence) for sequence in eng_sequences
)

max_tam_len = max(
    len(sequence) for sequence in tam_sequences
)

eng_sequences = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding="post"
)

tam_sequences = pad_sequences(
    tam_sequences,
    maxlen=max_tam_len,
    padding="post"
)

decoder_input = tam_sequences[:, :-1]

decoder_output = tam_sequences[:, 1:]

eng_vocab_size = len(
    eng_tokenizer.word_index
) + 1

tam_vocab_size = len(
    tam_tokenizer.word_index
) + 1

embedding_dim = 128
units = 256

encoder_inputs = Input(
    shape=(max_eng_len,),
    name="encoder_input"
)

encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=embedding_dim,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    units,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

decoder_inputs = Input(
    shape=(max_tam_len - 1,),
    name="decoder_input"
)

decoder_embedding = Embedding(
    input_dim=tam_vocab_size,
    output_dim=embedding_dim,
    name="decoder_embedding"
)(decoder_inputs)

decoder_lstm = LSTM(
    units,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

attention_layer = Attention(
    name="attention_layer"
)

context_vector = attention_layer([
    decoder_outputs,
    encoder_outputs
])


# Combine decoder output + attention
combined_output = Concatenate(
    axis=-1,
    name="attention_output"
)([
    decoder_outputs,
    context_vector
])
output_layer = Dense(
    tam_vocab_size,
    activation="softmax",
    name="output_layer"
)

outputs = output_layer(
    combined_output
)

model = Model(
    inputs=[
        encoder_inputs,
        decoder_inputs
    ],
    outputs=outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

model.fit(
    [eng_sequences, decoder_input],
    np.expand_dims(decoder_output, -1),
    epochs=300,
    batch_size=4,
    verbose=1
)

reverse_tam_word_index = {
    value: key
    for key, value in tam_tokenizer.word_index.items()
}

def translate(sentence):

    # Convert English sentence to tokens
    sequence = eng_tokenizer.texts_to_sequences(
        [sentence.lower()]
    )

    # Check unknown input
    if not sequence[0]:
        return "Sorry, this sentence is not in my vocabulary."

    # Padding
    sequence = pad_sequences(
        sequence,
        maxlen=max_eng_len,
        padding="post"
    )

    # Get <start> token
    start_token = tam_tokenizer.word_index["<start>"]

    # Start decoder with <start>
    decoder_sequence = [start_token]

    # Generate Tamil words one by one
    for _ in range(max_tam_len - 1):

        decoder_input_seq = pad_sequences(
            [decoder_sequence],
            maxlen=max_tam_len - 1,
            padding="post"
        )

        prediction = model.predict(
            [sequence, decoder_input_seq],
            verbose=0
        )

        # Get next token
        next_token = np.argmax(
            prediction[
                0,
                len(decoder_sequence) - 1
            ]
        )

        # Stop if padding
        if next_token == 0:
            break

        # Convert token to word
        next_word = reverse_tam_word_index.get(
            next_token,
            ""
        )

        # Stop at <end>
        if next_word == "<end>":
            break

        # Ignore start token
        if next_word != "<start>":
            decoder_sequence.append(next_token)

    # Convert tokens to Tamil words
    translated_words = []

    for token in decoder_sequence[1:]:

        word = reverse_tam_word_index.get(
            token,
            ""
        )

        if word not in ["", "<start>", "<end>"]:
            translated_words.append(word)

    return " ".join(translated_words)


# ============================================================
# 18. USER INPUT
# ============================================================

print("\n========================================")
print("   ENGLISH → TAMIL TRANSLATOR")
print("   Attention-Based NLP Model")
print("========================================")

while True:

    user_input = input(
        "\nEnter an English sentence "
        "(type 'exit' to stop): "
    )

    if user_input.lower().strip() == "exit":

        print("\nTranslator stopped.")
        break

    result = translate(user_input)

    print("Tamil Translation:", result)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, 5)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_input (InputLayer)    │ (None, 5)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_embedding (Embedding) │ (None, 5, 128)            │           4,352 │ encoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_embedding (Embedding) │ (None, 5, 128)            │           4,480 │ decoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 5, 256), (None,   │         394,240 │ encoder_embedding[0][0]    │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 5, 256), (None,   │         394,240 │ decoder_embedding[0][0],   │
│                               │ 256), (None, 256)]        │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_layer (Attention)   │ (None, 5, 256)            │               0 │ decoder_lstm[0][0],        │
│                               │                           │                 │ encoder_lstm[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_output              │ (None, 5, 512)            │               0 │ decoder_lstm[0][0],        │
│ (Concatenate)                 │                           │                 │ attention_layer[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_layer (Dense)          │ (None, 5, 35)             │          17,955 │ attention_output[0][0]     │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 815,267 (3.11 MB)

 Trainable params: 815,267 (3.11 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy: 0.1733 - loss: 3.5378   
Epoch 2/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2533 - loss: 3.4257 
Epoch 3/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2533 - loss: 3.1695 
Epoch 4/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2533 - loss: 2.6022
Epoch 5/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2667 - loss: 2.5527
Epoch 6/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.2267 - loss: 2.4234 
Epoch 7/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.3867 - loss: 2.3536
Epoch 8/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2800 - loss: 2.2653
Epoch 9/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.2800 - loss: 2.2169
Epoch 10/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4133 - loss: 2.1337 
Epoch 11/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4667 - loss: 2.0692
Epoch 12/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5


Enter an English sentence (type 'exit' to stop):  What are you doing?


Tamil Translation: நீங்கள் என்ன செய்கிறீர்கள்



Enter an English sentence (type 'exit' to stop):  exit



Translator stopped.
